In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models

In [2]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

In [3]:
from pathlib import Path
train, val = tf.keras.utils.audio_dataset_from_directory(
    directory=str(Path.home() / ".cache" / "ml-notebooks" / "speech-commands"),
    validation_split=0.3,
    output_sequence_length=16000,
    batch_size=64,
    seed=seed,
    subset='both'
)

Found 6505 files belonging to 4 classes.
Using 4554 files for training.
Using 1951 files for validation.


In [4]:
label = np.array(train.class_names)

In [5]:
def squeeze(a, l):
    a = tf.squeeze(a, axis=-1)
    return a, l

def get_st(wf):
    st = tf.signal.stft(wf, 255, 128)
    st = tf.abs(st)
    st = st[..., tf.newaxis]
    return st

def make_st(ds):
    return ds.map(
        lambda i, j: (get_st(i), j),
        tf.data.AUTOTUNE
    )

In [6]:
train = train.map(squeeze, tf.data.AUTOTUNE)
val = val.map(squeeze, tf.data.AUTOTUNE)

test = val.shard(num_shards=2, index=0)
val = val.shard(num_shards=2, index=1)

In [7]:
train_st = make_st(train)
val_st = make_st(val)
test_st = make_st(test)

In [8]:
for i, j in train_st.take(1):
    break

input_shape = i.shape[1:]
label_len = len(label)